In [1]:
# Cell 1: Imports
import dspy
from typing import List, Optional, Literal
from pydantic import BaseModel, Field
from docling.document_converter import DocumentConverter
import json
import os

In [30]:


# Cell 2: Configure DSPy LM
from openai import OpenAI

lm = dspy.LM(
    model="openai/gpt-5-mini",
    api_key=os.environ["OPENAI_API_KEY"],
)
dspy.configure(lm=lm)


In [18]:

# Cell 3: Define Output Schema with Descriptive Names
class ClimatePolicy(BaseModel):
    """Complete climate policy with extraction, validation, and classification"""
    
    # EXTRACTION FIELDS - What was found in the document
    policy_commitment: str = Field(
        description="A concise, self-contained summary rewritten as a clear commitment statement"
    )
    source_quote: str = Field(
        description="Verbatim text from source document (2-3 sentences, no paraphrasing)"
    )
    
    # SOUNDNESS INDICATORS - Quality markers from extraction
    has_measurable_target: str = Field(
        description="'Yes' or 'No'. If Yes, details like '40% reduction by 2030'"
    )
    has_deadline: str = Field(
        description="'Yes' or 'No'. If Yes, details like 'by 2030'"
    )
    has_legal_mandate: str = Field(
        description="'Yes' or 'No'. If Yes, details like 'regulatory requirement', 'ordinance'"
    )
    has_geographic_scope: str = Field(
        description="'Yes' or 'No'. If Yes, details like 'citywide', 'national', 'district-level'"
    )
    
    # VALIDATION RESULTS - Is this policy sound?
    is_actionable: bool = Field(
        description="True if policy is valid and actionable, False if vague/performative"
    )
    soundness_confidence: float = Field(
        description="Confidence in soundness assessment (0.0-1.0)"
    )
    validation_reasoning: str = Field(
        description="Why this policy is/isn't sound based on protocol criteria"
    )
    weak_language_detected: str = Field(
        description="Vague words like 'promote', 'explore', 'encourage' found in policy"
    )
    strong_language_detected: str = Field(
        description="Action words like 'mandate', 'install', 'phase-out', specific numbers"
    )
    
    # CLASSIFICATION FIELDS - What type of climate policy is this?
    climate_strategy_type: str = Field(
        description="Primary category: Mitigation, Adaptation, Resource Efficiency, or Nature-Based Solutions"
    )
    additional_strategy_types: List[str] = Field(
        default_factory=list,
        description="Other applicable categories if policy serves multiple purposes"
    )
    causal_mechanism: str = Field(
        description="How this policy achieves climate impact (e.g., 'Climate hazard → ↓ exposure')"
    )
    policy_instruments: List[str] = Field(
        default_factory=list,
        description="Specific mechanisms: renewable mandates, building codes, wetland restoration, etc."
    )
    classification_signals: List[str] = Field(
        default_factory=list,
        description="Key phrases that determined classification: 'flood defenses', 'renewables', etc."
    )
    classification_confidence: float = Field(
        description="Confidence in classification (0.0-1.0)"
    )
    co_benefits_identified: List[str] = Field(
        default_factory=list,
        description="Secondary benefits not used for primary classification"
    )
    
    # METADATA
    extraction_notes: str = Field(
        description="Any flags, edge cases, or ambiguities during extraction"
    )
    classification_notes: str = Field(
        description="Edge cases or ambiguities during classification, if any"
    )


class DocumentLocation(BaseModel):
    """Geographic context for the policy document"""
    country: str
    state_or_province: Optional[str] = None
    city: Optional[str] = None

In [19]:


# Cell 4: Extraction Signature and Module
class PolicyExtractionSignature(dspy.Signature):
    """
    Extract climate policies from document text.
    
    DEFINITION OF A POLICY
    A policy is a STATED COMMITMENT by a governing body to achieve a defined 
    outcome through deliberate action, resource allocation, or regulatory change.
    
    A policy is NOT:
    - Background information or problem descriptions
    - Statements of current conditions
    - Aspirations without any specified action
    - Descriptions of what other actors might do
    
    WHAT MAKES SOMETHING EXTRACTABLE
    Extract a statement as a policy if it contains AT LEAST ONE of:
    
    1. QUANTIFIABLE TARGET: Numbers with units and/or deadlines
       - "Reduce emissions 40% by 2030"
       - "Install 5GW of solar capacity"
       - "Retrofit 10,000 buildings"
    
    2. BINDING MECHANISM: Legal or regulatory force
       - "Mandatory building codes requiring..."
       - "Ban on single-use plastics"
       - "Carbon tax of $25/ton"
    
    3. SPECIFIC INTERVENTION: Named program, technology, or action
       - "National E-Mobility Program"
       - "Phase-out of coal-fired power plants"
       - "Expansion of bus rapid transit network"
    
    4. RESOURCE ALLOCATION: Committed funding or investment
       - "$500M allocated for renewable energy"
       - "Dedicated climate adaptation fund"
    
    EDGE CASES: EXTRACT BUT FLAG
    - Vague commitments ("promote", "encourage", "explore") — extract if 
      they reference a specific sector or mechanism, note vagueness
    - Strategies/roadmaps without targets — extract if they name concrete 
      action areas
    - Conditional commitments ("subject to international funding") — extract,
      note conditionality
    
    DO NOT EXTRACT
    - Pure context: "Climate change threatens coastal areas"
    - Current state: "The country currently has 2GW of wind capacity"  
    - Process statements: "Stakeholder consultations will be held"
    - Vague aspirations with no anchor: "We aspire to a green future"
    
    If no policies are found in the chunk, return an empty list.
    """
    document_text: str = dspy.InputField(
        desc="Text extracted from a climate policy document"
    )
    document_location: DocumentLocation = dspy.InputField(
        desc="Geographic context: country, state/province, city"
    )
    
    policies: List[dict] = dspy.OutputField(
        desc="List of extracted policies with metadata; empty list if none found"
    )


class PolicyExtractor(dspy.Module):
    """Extracts structured policy objects from document text"""
    
    def __init__(self):
        super().__init__()
        self.extract = dspy.ChainOfThought(PolicyExtractionSignature)
    
    def forward(self, document_text: str, document_location: DocumentLocation):
        result = self.extract(
            document_text=document_text,
            document_location=document_location
        )
        return result.policies


In [ ]:

# Cell 5: Validation Signature and Module
class PolicyValidationSignature(dspy.Signature):
    """
    Evaluate whether a climate policy is VALID (actionable and measurable) or NON-SOUND (vague and performative).

    The core of distinguishing between "rhetoric" and "commitment" is recognizing three dimensions: Specificity, Tangibility, and Scalability.

    A VALID POLICY contains Observable Parameters:
    
    1. Quantifiable Targets: The policy includes numbers, dates, and units.
       Example: "Retrofit 5,000 public buildings for 15% energy efficiency by 2030."
    
    2. Binding Mechanisms: The policy is backed by law or regulation, not just "encouragement."
       Example: "Implementing a mandatory 10% biofuel blend for commercial transport by 2027."
    
    3. Spatial Specificity: It identifies exactly where the impact will occur.
       Example: "Restoring 500 hectares of mangroves in the [X] River Delta."
    
    4. Technological Shift: Specifies a transition from one state to another.
       Example: "Phasing out all coal-fired power plants with a capacity over 100MW by 2040."

    A NON-SOUND POLICY is Performative or Vague:
    
    1. Process without Outcome: Focuses on meetings rather than results.
       Example: "Strengthening dialogue with environmental NGOs" or "Promoting awareness of climate change."
    
    2. Aspirational Language: Uses verbs like "seek to," "explore," "strive," or "aim" without a roadmap.
       Example: "We aim to explore the possibility of hydrogen power in the future."
    
    3. Lack of Baseline: Offers change without stating the starting point.
       Example: "We will increase green spaces" (without saying by how much or where).
    
    4. Administrative Maintenance: Standard government operations rebranded as climate action.
       Example: "Maintaining the existing Ministry of Environment budget."


    This evaluation shouldn't be too strict but make sure if a polcy exhbitis red flag to note it clearly, but if a policy exhbiits STRONG soundness in at least one of the dimensions it should be marked as VALID.
    """
    
    policy_statement: str = dspy.InputField(desc="The climate policy statement to evaluate")
    verbatim_text: str = dspy.InputField(desc="The original verbatim text from the source document")
    has_quantifiable_target: str = dspy.InputField(desc="Whether policy includes quantifiable targets")
    has_timeline: str = dspy.InputField(desc="Whether policy includes specific timeline/deadline")
    has_binding_mechanism: str = dspy.InputField(desc="Whether policy has legal/regulatory backing")
    has_spatial_specificity: str = dspy.InputField(desc="Whether policy identifies specific location/scope")
    
    validation_result: Literal["VALID", "NON-SOUND"] = dspy.OutputField(
        desc="Final classification: VALID (actionable/measurable) or NON-SOUND (vague/performative)"
    )
    
    confidence_score: float = dspy.OutputField(
        desc="Confidence in validation (0.0 to 1.0), where 1.0 is highest confidence"
    )
    
    reasoning: str = dspy.OutputField(
        desc="Detailed explanation referencing specific validation criteria and red flags"
    )
    
    weak_signals: str = dspy.OutputField(
        desc="Any red flag keywords detected (e.g., 'promote', 'explore', 'awareness', 'workshop')"
    )
    
    strong_signals: str = dspy.OutputField(
        desc="Any valid markers detected (e.g., percentages, MW, hectares, 'mandate', 'install', 'phase-out')"
    )
    final_verdict: bool = dspy.OutputField(
        desc="Based on the confidence score and validation result, is this policy likely to be sound?"
    )


class PolicyValidator(dspy.Module):
    """DSPy module for validating climate policy soundness"""
    
    def __init__(self):
        super().__init__()
        self.validate = dspy.ChainOfThought(PolicyValidationSignature)
    
    def forward(self, policy_data: dict):
        result = self.validate(
            policy_statement=policy_data.get("policy_statement", ""),
            verbatim_text=policy_data.get("verbatim_text", ""),
            has_quantifiable_target=policy_data.get("has_quantifiable_target", "Unknown"),
            has_timeline=policy_data.get("has_timeline", "Unknown"),
            has_binding_mechanism=policy_data.get("has_binding_mechanism", "Unknown"),
            has_spatial_specificity=policy_data.get("has_spatial_specificity", "Unknown")
        )
        return result


In [28]:

# Cell 6: Classification Signature and Module
class PolicyClassificationSignature(dspy.Signature):
    """Classify a climate policy into one or more categories based on its primary causal pathway.
    
    CATEGORY DEFINITIONS:
    
    1. MITIGATION - Acts on the climate system itself
       
       Definition: Mitigation policies are designed to influence the climate system by changing the amount 
       of greenhouse gases humans release into the atmosphere or by increasing the ability of natural or 
       engineered systems to remove those gases from the atmosphere. These are forward-looking policies 
       concerned with preventing future climate change rather than responding to impacts already occurring.
       
       Primary pathway: Human activity leads to DECREASED GHG emissions OR INCREASED carbon sinks, 
       which leads to DECREASED atmospheric forcing
       
       What to look for:
       - Direct or indirect emissions reduction targets
       - Energy system transformation (fossil fuels to renewables)
       - Decarbonization targets or net-zero commitments
       - Carbon sequestration framed in climate terms
       - Success measured by emissions reduced or carbon sequestered
       
       Typical mechanisms: renewable energy mandates, electrification (transport/buildings), 
       carbon pricing/offsets/caps, fuel switching, carbon capture and storage (CCS)
       
    2. ADAPTATION - Responds to climate impacts and builds resilience
       
       Definition: Adaptation policies reduce the negative consequences of climate change by helping 
       people, ecosystems, and infrastructure cope with current or expected climate impacts. Unlike 
       mitigation, adaptation does not attempt to slow climate change itself—it assumes climate change 
       is happening and focuses on reducing harm. Resilience is core: the capacity to continue functioning 
       under climate stress and recover quickly after disruption.
       
       Primary pathway: Climate hazard leads to DECREASED exposure OR DECREASED sensitivity OR 
       INCREASED adaptive capacity
       
       What to look for:
       - Explicit reference to climate impacts (heat, flooding, drought, sea-level rise, storms)
       - Resilience, preparedness, or risk reduction language
       - Protection of people, assets, or systems from future climate conditions
       - Success measured by reduced losses, maintained services, avoided disruption, increased preparedness
       
       Typical mechanisms: flood defenses, heat action plans, climate-resilient infrastructure, 
       emergency response capacity, insurance/risk-transfer instruments, early warning systems
       
    3. RESOURCE EFFICIENCY - Optimizes resource use regardless of climate
       
       Definition: Resource efficiency policies deliver the same level of service, output, or quality 
       of life while using fewer physical resources (energy, water, materials, land). The core idea is 
       that systems are inefficient and consume more than necessary. These policies can be fully understood 
       and justified even in the absence of climate change concerns—efficiency is the goal, not climate impact.
       
       Primary pathway: Same service leads to DECREASED energy/water/material input OR DECREASED waste output
       
       What to look for:
       - Efficiency improvements that don't require climate justification
       - Optimization of inputs, flows, or lifecycles
       - Performance-based standards and benchmarks
       - Success measured by resource savings per unit of output
       - Emissions reduction is a secondary consequence, NOT the primary goal
       
       Typical mechanisms: energy efficiency codes, water efficiency standards, circular economy policies, 
       waste reduction/recycling mandates, industrial process optimization, building performance standards
       
    4. NATURE-BASED SOLUTIONS (NbS) - Uses ecosystems as infrastructure
       
       Definition: Nature-based solutions intentionally use natural systems or ecological processes to 
       address environmental, climate, or societal challenges. Effectiveness depends on the functioning 
       of living systems rather than solely on engineered infrastructure. These are defined by their 
       MECHANISM (ecosystems), not their outcome. If removing the ecological element would fundamentally 
       break the policy's effectiveness, it's NbS.
       
       Primary pathway: Ecosystem protection/restoration leads to climate mitigation OR resilience 
       OR broader co-benefits
       
       What to look for:
       - Explicit use of ecosystems as infrastructure
       - Restoration, conservation, or enhancement of natural systems
       - Solutions that rely on biological processes (carbon uptake, water absorption, cooling)
       - Policy effectiveness depends on living systems functioning properly
       
       Typical mechanisms: urban forests, wetland restoration, green roofs and corridors, 
       mangroves, coastal dunes, riparian buffers, soil restoration
    
    CLASSIFICATION RULES:
    
    Rule 1: Primary mechanism > secondary benefits
    - Classify by what the policy DOES, not by claimed co-benefits
    - Example: "Energy efficiency to reduce emissions" → Resource Efficiency (NOT Mitigation)
    
    Rule 2: Climate system vs climate impacts
    - Acts on emissions/carbon cycle → Mitigation
    - Acts on hazards/vulnerability → Adaptation
    
    Rule 3: Natural vs engineered pathways
    - Ecosystem-mediated → NbS
    - Technological/infrastructural → other categories
    
    Rule 4: Multi-label is allowed
    - Policies can be Mitigation + NbS, Adaptation + NbS, etc.
    - But the PRIMARY label must be identifiable
    
    EDGE CASES:
    
    Green roofs:
    - Urban heat exposure/stormwater management → Adaptation
    - Ecosystem restoration → NbS
    - Carbon storage → Mitigation (secondary)
    
    Passive cooling/shading:
    - Protecting occupants during heatwaves → Adaptation
    - Reducing energy demand → Resource Efficiency
    
    District energy:
    - Decarbonizing heating/cooling → Mitigation
    - Energy reliability during extreme weather → Adaptation
    
    Floodable parks/wetlands:
    - Reducing flood risk through natural absorption → NbS + Adaptation
    - Biodiversity enhancement (no climate risk) → NbS only
    
    Infrastructure hardening:
    - Strengthening grids/systems for storms/heat → Adaptation
    - Not mitigation unless emissions reduction is main objective
    """
    
    policy_statement: str = dspy.InputField(desc="The climate policy statement to classify")
    verbatim_text: str = dspy.InputField(desc="Original verbatim text from source document")
    context: str = dspy.InputField(desc="Additional context about the policy (optional)", default="")
    
    primary_category: str = dspy.OutputField(
        desc="The PRIMARY category: Mitigation, Adaptation, Resource Efficiency, or Nature-Based Solutions"
    )
    
    secondary_categories: str = dspy.OutputField(
        desc="Additional applicable categories as comma-separated list, or 'None' if single-category"
    )
    
    primary_causal_pathway: str = dspy.OutputField(
        desc="The policy's primary causal pathway (e.g., 'Climate hazard → ↓ exposure')"
    )
    
    key_indicators: str = dspy.OutputField(
        desc="Specific words/phrases that signaled this classification (e.g., 'flood defenses', 'renewables mandate', 'ecosystem restoration')"
    )
    
    policy_mechanisms: str = dspy.OutputField(
        desc="The specific mechanisms used (e.g., 'renewable energy mandates', 'heat action plans', 'wetland restoration')"
    )
    
    classification_reasoning: str = dspy.OutputField(
        desc="Detailed step-by-step explanation of why this classification was chosen, referencing specific rules"
    )
    
    confidence_score: float = dspy.OutputField(
        desc="Confidence in classification (0.0 to 1.0)"
    )
    
    edge_case_notes: str = dspy.OutputField(
        desc="Any edge case considerations or ambiguities encountered, or 'None'"
    )
    
    co_benefits: str = dspy.OutputField(
        desc="Secondary benefits mentioned but not used for primary classification (e.g., 'emissions reduction', 'biodiversity'), or 'None'"
    )


class PolicyClassifier(dspy.Module):
    """DSPy module for climate policy classification"""
    
    def __init__(self):
        super().__init__()
        self.classify = dspy.ChainOfThought(PolicyClassificationSignature)
    
    def forward(self, policy_data: dict):
        result = self.classify(
            policy_statement=policy_data.get("policy_statement", ""),
            verbatim_text=policy_data.get("verbatim_text", "")
        )
        return result


In [ ]:

# Cell 7: Main Pipeline Function
def process_climate_policy_document(
    pdf_path: str,
    country: str,
    state_or_province: Optional[str] = None,
    city: Optional[str] = None,
    output_prefix: str = "output"
) -> tuple[List[ClimatePolicy], dict]:
    """
    Complete pipeline: Extract → Validate → Classify climate policies from a PDF document.
    
    Args:
        pdf_path: Path to the PDF file
        country: Country name
        state_or_province: State or province name (optional)
        city: City name (optional)
        output_prefix: Prefix for output JSON files
    
    Returns:
        (list of ClimatePolicy objects, statistics dict)
    """
    
    print(f"\n{'='*70}")
    print(f"PROCESSING: {pdf_path}")
    print(f"Location: {city or country}, {state_or_province or ''}, {country}")
    print(f"{'='*70}\n")
    
    # Step 1: Convert PDF to markdown
    print("Step 1: Converting PDF to markdown...")
    converter = DocumentConverter()
    document = converter.convert(pdf_path)
    markdown_text = document.document.export_to_markdown()
    print(f"✓ Converted {len(markdown_text)} characters\n")
    
    # Step 2: Extract policies
    print("Step 2: Extracting policies...")
    location = DocumentLocation(
        country=country,
        state_or_province=state_or_province,
        city=city
    )
    
    extractor = PolicyExtractor()
    extracted_policies = extractor(
        document_text=markdown_text,
        document_location=location
    )
    print(f"✓ Extracted {len(extracted_policies)} policy candidates\n")
    
    # Step 3: Validate policies
    print("Step 3: Validating policy soundness...")
    validator = PolicyValidator()
    valid_policies = []
    
    for policy_data in extracted_policies:
        validation_result = validator.forward(policy_data)
        
        if validation_result.validation_result == "VALID" and validation_result.final_verdict:
            policy_data['validation'] = {
                'is_valid': True,
                'confidence': validation_result.confidence_score,
                'reasoning': validation_result.reasoning,
                'weak_signals': validation_result.weak_signals,
                'strong_signals': validation_result.strong_signals
            }
            valid_policies.append(policy_data)
    
    print(f"✓ {len(valid_policies)} policies passed validation\n")
    
    # Step 4: Classify valid policies
    print("Step 4: Classifying policies...")
    classifier = PolicyClassifier()
    final_policies = []
    
    for policy_data in valid_policies:
        classification_result = classifier.forward(policy_data)
        
        # Parse lists from comma-separated strings
        secondary_cats = [c.strip() for c in classification_result.secondary_categories.split(",") 
                         if c.strip() and c.strip().lower() != "none"]
        key_inds = [k.strip() for k in classification_result.key_indicators.split(",") if k.strip()]
        mechanisms = [m.strip() for m in classification_result.policy_mechanisms.split(",") if m.strip()]
        co_bens = [b.strip() for b in classification_result.co_benefits.split(",") 
                  if b.strip() and b.strip().lower() != "none"]
        
        # Create ClimatePolicy object with descriptive field names
        climate_policy = ClimatePolicy(
            # Extraction
            policy_commitment=policy_data.get("policy_statement", ""),
            source_quote=policy_data.get("verbatim_text", ""),
            has_measurable_target=policy_data.get("has_quantifiable_target", "Unknown"),
            has_deadline=policy_data.get("has_timeline", "Unknown"),
            has_legal_mandate=policy_data.get("has_binding_mechanism", "Unknown"),
            has_geographic_scope=policy_data.get("has_spatial_specificity", "Unknown"),
            extraction_notes=policy_data.get("extraction_rationale", ""),
            
            # Validation
            is_actionable=policy_data['validation']['is_valid'],
            soundness_confidence=policy_data['validation']['confidence'],
            validation_reasoning=policy_data['validation']['reasoning'],
            weak_language_detected=policy_data['validation']['weak_signals'],
            strong_language_detected=policy_data['validation']['strong_signals'],
            
            # Classification
            climate_strategy_type=classification_result.primary_category,
            additional_strategy_types=secondary_cats,
            causal_mechanism=classification_result.primary_causal_pathway,
            policy_instruments=mechanisms,
            classification_signals=key_inds,
            classification_confidence=classification_result.confidence_score,
            co_benefits_identified=co_bens,
            classification_notes=classification_result.edge_case_notes
        )
        
        final_policies.append(climate_policy)
    
    print(f"✓ Classified {len(final_policies)} policies\n")
    
    # Step 5: Calculate statistics
    stats = {
        'total_extracted': len(extracted_policies),
        'passed_validation': len(valid_policies),
        'final_classified': len(final_policies),
        'validation_rate': len(valid_policies) / len(extracted_policies) if extracted_policies else 0,
        'category_distribution': {},
        'avg_classification_confidence': 0.0,
        'multi_category_count': 0
    }
    
    for policy in final_policies:
        cat = policy.climate_strategy_type
        stats['category_distribution'][cat] = stats['category_distribution'].get(cat, 0) + 1
        stats['avg_classification_confidence'] += policy.classification_confidence
        if policy.additional_strategy_types:
            stats['multi_category_count'] += 1
    
    if final_policies:
        stats['avg_classification_confidence'] /= len(final_policies)
    
    # Step 6: Save outputs
    output_file = f\"../data/policies/{output_prefix}_policies.json\"
    with open(output_file, \"w\") as f:
        json.dump([policy.dict() for policy in final_policies], f, indent=2)
    
    stats_file = f\"../data/stats/{output_prefix}_stats.json\"
    with open(stats_file, "w") as f:
        json.dump(stats, f, indent=2)
    
    print(f"{'='*70}")
    print("PIPELINE COMPLETE")
    print(f"{'='*70}")
    print(f"Results saved to: {output_file}")
    print(f"Statistics saved to: {stats_file}")
    print(f"\nSummary:")
    print(f"  Extracted: {stats['total_extracted']} policies")
    print(f"  Valid: {stats['passed_validation']} policies ({stats['validation_rate']:.1%})")
    print(f"  Multi-category: {stats['multi_category_count']} policies")
    print(f"  Avg confidence: {stats['avg_classification_confidence']:.3f}")
    print(f"\nCategory Distribution:")
    for cat, count in sorted(stats['category_distribution'].items(), key=lambda x: x[1], reverse=True):
        pct = (count / len(final_policies) * 100) if final_policies else 0
        print(f"    {cat}: {count} ({pct:.1f}%)")
    
    return final_policies, stats


In [ ]:

# Cell 8: Example Usage
# Process a single document
policies, stats = process_climate_policy_document(
    pdf_path="../pdfs/Seattle.pdf",
    country="United States",
    state_or_province="Washington",
    city="Seattle",
    output_prefix="seattle"
)

In [24]:


# Cell 9: Batch Processing Multiple Documents
def process_multiple_documents(document_configs: dict):
    """
    Process multiple documents in batch.
    
    Args:
        document_configs: Dict mapping output_prefix to config dict with keys:
            - pdf_path: str
            - country: str
            - state_or_province: Optional[str]
            - city: Optional[str]
    """
    all_results = {}
    
    for prefix, config in document_configs.items():
        try:
            policies, stats = process_climate_policy_document(
                pdf_path=config['pdf_path'],
                country=config['country'],
                state_or_province=config.get('state_or_province'),
                city=config.get('city'),
                output_prefix=prefix
            )
            all_results[prefix] = {
                'policies': policies,
                'stats': stats,
                'status': 'success'
            }
        except Exception as e:
            print(f"ERROR processing {prefix}: {e}")
            all_results[prefix] = {
                'status': 'error',
                'error': str(e)
            }
    
    return all_results

# Example batch processing
documents_to_process = {
    "seattle": {
        "pdf_path": "../pdfs/Seattle.pdf",
        "country": "United States",
        "state_or_province": "Washington",
        "city": "Seattle"
    },
    "las_vegas": {
        "pdf_path": "../pdfs/CLV.pdf",
        "country": "United States",
        "state_or_province": "Nevada",
        "city": "Las Vegas"
    },
    "miami_dade": {
        "pdf_path": "../pdfs/MIAMI_DADE.pdf",
        "country": "United States",
        "state_or_province": "Florida",
        "city": "Miami-Dade"
    },
    "portugal": {
        "pdf_path": "../pdfs/2021 Portugal ADCOM_UNFCCC.pdf",
        "country": "Portugal",
        "state_or_province": None,
        "city": None
    },
    "austin":{
        'pdf_path': "../pdfs/Austin_climate_equity.pdf",
        "country": "United States",
        "state_or_province": "Texas",
        "city": "Austin"
    },
    "dakar":{
        'pdf_path': "../pdfs/Dakar.pdf",
        "country": "Senegal",
        "state_or_province": None,
        "city": "Dakar"
    },
    "kuwait":{
        'pdf_path': "../pdfs/Kuwait-NAP-2019-2030.pdf",
        "country": "Kuwait",
        "state_or_province": None,
        "city": None
    },
    "Chicago":{
        'pdf_path': "../pdfs/Chicago-CAP-071822.pdf",
        "country": "United States",
        "state_or_province": "Illinois",
        "city": "Chicago"
    }
}

# Process all documents
# results = process_multiple_documents(documents_to_process)

In [29]:
results = process_multiple_documents(documents_to_process)


2026-01-20 23:02:16,777 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-01-20 23:02:16,792 - INFO - Going to convert document batch...
2026-01-20 23:02:16,793 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e15bc6f248154cc62f8db15ef18a8ab7
2026-01-20 23:02:16,794 - INFO - Auto OCR model selected ocrmac.
2026-01-20 23:02:16,794 - INFO - Accelerator device: 'mps'



PROCESSING: pdfs/Seattle.pdf
Location: Seattle, Washington, United States

Step 1: Converting PDF to markdown...


2026-01-20 23:02:18,436 - INFO - Accelerator device: 'mps'
2026-01-20 23:02:19,550 - INFO - Processing document Seattle.pdf
2026-01-20 23:02:30,458 - INFO - Finished converting document Seattle.pdf in 13.69 sec.
2026/01/20 23:02:30 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.


✓ Converted 49889 characters

Step 2: Extracting policies...
✓ Extracted 25 policy candidates

Step 3: Validating policy soundness...


2026/01/20 23:02:42 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/20 23:02:42 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/20 23:02:42 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/20 23:02:43 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/20 23:02:43 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/20 23:02:43 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/20 23:02:43 WARNING dspy.primitives.module: Calling modu

✓ 0 policies passed validation

Step 4: Classifying policies...
✓ Classified 0 policies

PIPELINE COMPLETE
Results saved to: seattle_policies.json
Statistics saved to: seattle_stats.json

Summary:
  Extracted: 25 policies
  Valid: 0 policies (0.0%)
  Multi-category: 0 policies
  Avg confidence: 0.000

Category Distribution:

PROCESSING: pdfs/CLV.pdf
Location: Las Vegas, Nevada, United States

Step 1: Converting PDF to markdown...


2026-01-20 23:02:45,385 - INFO - Accelerator device: 'mps'
2026-01-20 23:02:46,429 - INFO - Processing document CLV.pdf


KeyboardInterrupt: 

In [12]:
# Cell 10: Export Results to CSV
import pandas as pd

def export_results_to_csv(results: dict, output_filename: str = "all_policies.csv"):
    """
    Export batch processing results to a CSV file.
    
    Args:
        results: Output from process_multiple_documents()
        output_filename: Name of the output CSV file
    """
    all_policies_data = []
    
    for location_name, result in results.items():
        if result['status'] == 'success':
            policies = result['policies']
            
            for policy in policies:
                # Convert policy to dict if it's a Pydantic model
                policy_dict = policy.dict() if hasattr(policy, 'dict') else policy
                
                # Flatten the data for CSV
                row = {
                    'location': location_name,
                    'policy_commitment': policy_dict.get('policy_commitment', ''),
                    'source_quote': policy_dict.get('source_quote', ''),
                    
                    # Soundness indicators
                    'has_measurable_target': policy_dict.get('has_measurable_target', ''),
                    'has_deadline': policy_dict.get('has_deadline', ''),
                    'has_legal_mandate': policy_dict.get('has_legal_mandate', ''),
                    'has_geographic_scope': policy_dict.get('has_geographic_scope', ''),
                    
                    # Validation
                    'is_actionable': policy_dict.get('is_actionable', ''),
                    'soundness_confidence': policy_dict.get('soundness_confidence', ''),
                    'validation_reasoning': policy_dict.get('validation_reasoning', ''),
                    'weak_language_detected': policy_dict.get('weak_language_detected', ''),
                    'strong_language_detected': policy_dict.get('strong_language_detected', ''),
                    
                    # Classification
                    'climate_strategy_type': policy_dict.get('climate_strategy_type', ''),
                    'additional_strategy_types': '; '.join(policy_dict.get('additional_strategy_types', [])),
                    'causal_mechanism': policy_dict.get('causal_mechanism', ''),
                    'policy_instruments': '; '.join(policy_dict.get('policy_instruments', [])),
                    'classification_signals': '; '.join(policy_dict.get('classification_signals', [])),
                    'classification_confidence': policy_dict.get('classification_confidence', ''),
                    'co_benefits_identified': '; '.join(policy_dict.get('co_benefits_identified', [])),
                    
                    # Notes
                    'extraction_notes': policy_dict.get('extraction_notes', ''),
                    'classification_notes': policy_dict.get('classification_notes', '')
                }
                
                all_policies_data.append(row)
    
    # Create DataFrame and export to CSV
    df = pd.DataFrame(all_policies_data)
    df.to_csv(output_filename, index=False, encoding='utf-8')
    
    print(f"\n{'='*70}")
    print(f"CSV EXPORT COMPLETE")
    print(f"{'='*70}")
    print(f"File: {output_filename}")
    print(f"Total policies exported: {len(df)}")
    print(f"\nPolicies by location:")
    print(df['location'].value_counts().to_string())
    print(f"\nPolicies by climate strategy type:")
    print(df['climate_strategy_type'].value_counts().to_string())
    
    return df

# Export to CSV
df = export_results_to_csv(results, "all_climate_policies.csv")

NameError: name 'results' is not defined